# 00 · Setup & Data
**Driver Monitoring System** — distracted + drowsy driver detection

This notebook:
1. Mounts Google Drive and clones the project repo
2. Downloads the State Farm dataset from Kaggle **once** (saved to Drive)
3. Extracts it to Colab's fast local disk
4. Runs exploratory data analysis
5. Creates the **driver-wise** train/val/test split and saves it to Drive

**Runtime:** a CPU runtime is enough here (save your GPU quota for training).
**Before running:** add your Kaggle credentials in Colab **Secrets** (see section 5) (key icon, left sidebar) with notebook access on, and join the State Farm competition on Kaggle.

## 1 · Settings — change `GITHUB_USER` to your GitHub username

In [ ]:
GITHUB_USER = "harshithdande28"   # <-- change this
REPO_NAME = "driver-monitoring-system"

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3 · Get the latest code and install extras

In [ ]:
import os, sys
REPO_DIR = f"/content/{REPO_NAME}"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 4 · Load config and create Drive folders

In [ ]:
from pathlib import Path
from src.config import load_config, class_names

cfg = load_config()
CLASS_NAMES = class_names(cfg)
SEED = cfg["seed"]

DRIVE_ROOT = Path(cfg["paths"]["drive_root"])
for sub in ["data", "checkpoints", "mlruns", "exports"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

ZIP_NAME = cfg["data"]["zip_name"]
DRIVE_ZIP = DRIVE_ROOT / "data" / ZIP_NAME     # permanent copy
LOCAL_ZIP = Path("/content") / ZIP_NAME         # temporary copy
LOCAL_DATA = Path(cfg["paths"]["local_data"])   # extracted images (fast disk)
SPLITS_FILE = Path(cfg["paths"]["splits_file"])
print("Drive folder:", DRIVE_ROOT)

## 5 · Download the dataset (first time only)
Downloads ~4 GB to Colab, then copies the zip to Drive so later sessions skip Kaggle entirely.

**Credentials (Colab Secrets, notebook access ON) — either:**
- **New Kaggle token** (shown as text in Kaggle Settings → API): one secret named `KAGGLE_API_TOKEN`, **or**
- **Legacy kaggle.json**: two secrets, `KAGGLE_USERNAME` and `KAGGLE_KEY`.

**403 Forbidden** → open the competition on Kaggle and accept the rules. **401 Unauthorized** → the token is wrong or expired; create a new one.


In [ ]:
import shutil
from google.colab import userdata

def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if DRIVE_ZIP.exists():
    print("Dataset zip already on Drive — skipping download.")
else:
    token = _secret("KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token.strip()
        print("Using new-style Kaggle API token.")
    else:
        os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME") or ""
        os.environ["KAGGLE_KEY"] = _secret("KAGGLE_KEY") or ""
        assert os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_KEY"], \
            "Add KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY) in Colab Secrets."
        print("Using legacy kaggle.json credentials.")
    !pip install -q -U kaggle   # recent version needed for new-style tokens
    !kaggle competitions download -c {cfg["data"]["kaggle_competition"]} -p /content
    assert LOCAL_ZIP.exists(), "Download failed — see the Kaggle error above."
    print("Copying zip to Drive (a few minutes)...")
    shutil.copy(LOCAL_ZIP, DRIVE_ZIP)
    print("Saved to", DRIVE_ZIP)


## 6 · Extract to local disk (every session, ~1–2 min)

In [ ]:
if (LOCAL_DATA / "driver_imgs_list.csv").exists():
    print("Already extracted.")
else:
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    zip_to_use = LOCAL_ZIP if LOCAL_ZIP.exists() else DRIVE_ZIP
    !unzip -q "{zip_to_use}" -d "{LOCAL_DATA}"
!ls {LOCAL_DATA}
!ls {LOCAL_DATA}/imgs/train

## 7 · Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt
from src.data.split import load_driver_list
from src.data.eda import plot_class_distribution, plot_images_per_driver, plot_sample_grid

df = load_driver_list(LOCAL_DATA)
missing = [p for p in df["path"] if not (LOCAL_DATA / p).exists()]
print(f"{len(df):,} labeled images | {df['subject'].nunique()} drivers | {df['classname'].nunique()} classes")
print(f"Missing image files: {len(missing)}")

plot_class_distribution(df, CLASS_NAMES); plt.show()
plot_images_per_driver(df); plt.show()

In [ ]:
plot_sample_grid(df, LOCAL_DATA, CLASS_NAMES, per_class=3, seed=SEED); plt.show()

## 8 · Driver-wise split
Entire **drivers** are assigned to train, val or test — never individual images.
Frames of one driver are nearly identical, so an image-level split would leak and inflate accuracy.

In [ ]:
from src.data.split import make_driver_split, summarize_split, class_counts_by_split, check_no_leakage

split_df = make_driver_split(
    df,
    val_fraction=cfg["data"]["val_fraction"],
    test_fraction=cfg["data"]["test_fraction"],
    seed=SEED,
    trials=cfg["data"]["split_search_trials"],
)
check_no_leakage(split_df)
print("✅ No driver appears in more than one split\n")
display(summarize_split(split_df))
display(class_counts_by_split(split_df))
for s in ["train", "val", "test"]:
    print(f"{s:5s} drivers:", sorted(split_df.loc[split_df.split == s, "subject"].unique()))

plot_images_per_driver(split_df); plt.show()

## 9 · Save the split to Drive

In [ ]:
SPLITS_FILE.parent.mkdir(parents=True, exist_ok=True)
split_df.to_csv(SPLITS_FILE, index=False)
print("Saved:", SPLITS_FILE)

## 10 · Run the project tests

In [ ]:
!python -m pytest -q

---
✅ **Done.** The dataset zip and `splits.csv` are on Drive, so every future notebook just extracts and loads them.
**Next:** `01_train_distraction.ipynb` (switch to a **T4 GPU** runtime).